In [1]:
# ============================================================
# CS781 Capstone - PHASE 2: Legal NER (strict character spans)
# Starts from YOUR Phase-1 model (best.pt) + token-classification head (BIO tags).
# One notebook for all experiments: change MODEL_TYPE / FT_METHOD in Cell 1.
#   FT_METHOD: "full" | "partial" | "bitfit" | "lora" | "lorma" | "adapter"
#   (knowledge distillation = separate notebook, after you have a teacher)
# Paste each "# %%" block into its own Kaggle cell (or run all in one cell).
# ============================================================

# %% 0. IMPORTS
import os, json, math, time, glob, random, zipfile, copy
from collections import defaultdict
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, PreTrainedTokenizerFast

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None

# %% 1. CONFIG  <-------------------------------------------------------- EDIT HERE
QUICK_TEST = False          # True = few docs, 1 epoch, to check the pipeline. False = real run.

MODEL_TYPE = "hybrid"       # "gpt" (Model 1) | "llama" (Model 2) | "gqa" (Model 3) | "hybrid" (Model 4)
PHASE1_CKPT = "/kaggle/input/datasets/kavyajain123987/phase1-models/model4_hybrid"         # full path to that model's best.pt, e.g. "/kaggle/input/phase1-model2/best.pt"
                           # (None -> prints every best.pt found under /kaggle/input so you can pick)
TOKENIZER = "/kaggle/input/datasets/kavyajain123987/phase1/tokenizer (1).json"         # MUST be the tokenizer that Phase-1 model was trained with:
                           # "gpt2", or the path to your legal tokenizer json

FT_METHOD = "adapter"         # "full" | "partial" | "bitfit" | "lora" | "lorma" | "adapter"
FREEZE_BOTTOM = 4          # partial: freeze embeddings + this many lowest blocks
LORA_R, LORA_ALPHA = 8, 16 # lora / lorma rank and scaling
ADAPTER_DIM = 64           # adapter bottleneck size

MAX_LEN = 512              # tokens per window (<= Phase-1 context length)
OVERLAP = 128              # overlap between consecutive windows (left context for the next window)
BATCH_SIZE = 8
EPOCHS = 6
LR_BODY = 1e-4             # LR for Phase-1 weights that are trained (full / partial)
LR_NEW = 1e-3              # LR for new parameters (head, LoRA, LoRMA, adapters, BitFit)
WEIGHT_DECAY = 0.01
O_WEIGHT = 1.0             # loss weight of the "O" class (<1 = push towards predicting entities)
VAL_FRAC = 0.15            # fraction of training documents held out for local strict-F1
SEED = 42

# ---- knowledge distillation (mandatory experiment) ----
KD_TEACHER = None          # path to a trained Phase-2 teacher (its best_ner.pt) -> turns on distillation
KD_TEACHER_TYPE = None     # teacher's MODEL_TYPE (None = same as MODEL_TYPE)
KD_TEACHER_CKPT = None     # teacher's Phase-1 best.pt (only to read its config; None = PHASE1_CKPT)
KD_KEEP_EVERY = 2          # student = Phase-1 model keeping every 2nd block (8 -> 4 layers)
KD_T, KD_ALPHA = 2.0, 0.5  # softmax temperature; loss = ALPHA * CE(gold) + (1-ALPHA) * T^2 * KL(teacher || student)

OUTPUT_DIR = f"/kaggle/working/phase2_{MODEL_TYPE}_{FT_METHOD}" + ("_kd" if KD_TEACHER else "")
os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

# %% 2. DOWNLOAD PHASE-2 DATA + LOOK AT IT
from huggingface_hub import snapshot_download
DATA_DIR = snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                             allow_patterns=["Phase_2_LNER/*"], local_dir="/tmp/cs781_data")
P2 = os.path.join(DATA_DIR, "Phase_2_LNER")
for root, _, files in os.walk(P2):
    for f in files:
        print(os.path.join(root, f), f"{os.path.getsize(os.path.join(root, f)) / 1e6:.1f} MB")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def peek(obj, n=300):
    s = json.dumps(obj, ensure_ascii=False)
    return s[:n] + ("..." if len(s) > n else "")

for name in ["Train/fold1-text.json", "Train/fold1-gold.json", "Test/fold3-text.json"]:
    d = load_json(os.path.join(P2, name))
    first = next(iter(d.items())) if isinstance(d, dict) else d[0]
    print(f"\n{name}: {type(d).__name__} with {len(d)} entries | first: {peek(first)}")
LABELS = [l.strip() for l in open(os.path.join(P2, "Train/labels.txt"), encoding="utf-8") if l.strip()]
print("\nLabels:", LABELS)

# %% 3. PARSE DATA (handles dict / list layouts) + DETECT END-OFFSET CONVENTION
def parse_texts(obj):
    """-> {doc_id(str): text}"""
    out = {}
    items = obj.items() if isinstance(obj, dict) else enumerate(obj)
    for k, v in items:
        if isinstance(v, str):
            out[str(k)] = v
        elif isinstance(v, dict):
            doc_id = str(v.get("id", v.get("doc_id", k)))
            out[doc_id] = v.get("text", v.get("document", ""))
    return out

def parse_ents(v):
    """one document's annotations -> list of (start, end, label, text_or_None)"""
    if isinstance(v, dict):
        v = v.get("entities", v.get("annotations", v.get("spans", [])))
    ents = []
    for e in v:
        if isinstance(e, (list, tuple)):
            ents.append((int(e[0]), int(e[1]), str(e[2]), e[3] if len(e) > 3 else None))
        elif isinstance(e, dict):
            ents.append((int(e.get("start", e.get("start_char"))), int(e.get("end", e.get("end_char"))),
                         str(e.get("label", e.get("type", e.get("entity")))), e.get("text")))
    return ents

def parse_gold(obj):
    out = {}
    items = obj.items() if isinstance(obj, dict) else enumerate(obj)
    for k, v in items:
        doc_id = str(v.get("id", v.get("doc_id", k))) if isinstance(v, dict) and ("id" in v or "doc_id" in v) else str(k)
        out[doc_id] = parse_ents(v)
    return out

docs, gold = {}, {}
for fold in ["fold1", "fold2"]:
    docs.update(parse_texts(load_json(os.path.join(P2, f"Train/{fold}-text.json"))))
    gold.update(parse_gold(load_json(os.path.join(P2, f"Train/{fold}-gold.json"))))
test_docs = parse_texts(load_json(os.path.join(P2, "Test/fold3-text.json")))
missing = [d for d in gold if d not in docs]
print(f"Train docs: {len(docs)} | docs with gold: {len(gold)} | gold docs without text: {len(missing)} | test docs: {len(test_docs)}")

# end offset: exclusive (text[s:e]) or inclusive (text[s:e+1])?  Decided from the entity text in gold.
n_excl = n_incl = n_txt = 0
for d, ents in gold.items():
    t = docs.get(d, "")
    for s, e, lab, et in ents:
        if et:
            n_txt += 1
            n_excl += t[s:e] == et
            n_incl += t[s:e + 1] == et
END_INCLUSIVE = n_incl > n_excl
print(f"Entity text matches: exclusive {n_excl}/{n_txt}, inclusive {n_incl}/{n_txt} "
      f"-> END_INCLUSIVE = {END_INCLUSIVE}")
if n_txt and max(n_excl, n_incl) < 0.95 * n_txt:
    print("WARNING: offsets don't match the text well - check the data format before training!")

label_count = defaultdict(int)
for ents in gold.values():
    for _, _, lab, _ in ents:
        label_count[lab] += 1
print("Entities per label:", dict(sorted(label_count.items(), key=lambda x: -x[1])))

# internal spans are always [start, end) exclusive
def to_excl(s, e):
    return (s, e + 1) if END_INCLUSIVE else (s, e)

def from_excl(s, e):
    return (s, e - 1) if END_INCLUSIVE else (s, e)

# %% 4. TOKENIZER (same one as the Phase-1 model)
if TOKENIZER == "gpt2":
    tokenizer = AutoTokenizer.from_pretrained("gpt2", token=HF_TOKEN)
else:
    tokenizer = PreTrainedTokenizerFast(tokenizer_file=TOKENIZER)
tokenizer.model_max_length = int(1e9)
print("Tokenizer vocab:", len(tokenizer))

TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
TAG2ID = {t: i for i, t in enumerate(TAGS)}

def tokenize_with_offsets(text):
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids, offs = enc["input_ids"], []
    for s, e in enc["offset_mapping"]:
        while s < e and text[s].isspace():   # byte-level BPE offsets can include the leading space
            s += 1
        offs.append((s, e))
    return ids, offs

def bio_tags(text, offs, ents):
    """char spans -> one BIO tag id per token"""
    char_ent = np.full(len(text) + 1, -1, dtype=np.int64)
    for j, (s, e, lab) in enumerate(ents):
        char_ent[s:e] = j
    tags, prev = [], -1
    for s, e in offs:
        if s >= e:                 # whitespace-only token: no label, doesn't break an entity
            tags.append(-100); continue
        j = char_ent[s]
        if j < 0:
            tags.append(TAG2ID["O"]); prev = -1
        else:
            tags.append(TAG2ID[("I-" if j == prev else "B-") + ents[j][2]]); prev = j
    return tags

STRIP_TRAIL = ""   # characters stripped from the end of decoded spans; chosen in the oracle check below

def decode_spans(text, offs, tags):
    """BIO tag ids -> list of (start, end, label), end exclusive"""
    spans, cur = [], None
    for (s, e), t in zip(offs, tags):
        if s >= e:                 # whitespace-only token: skip without closing the current span
            continue
        tag = TAGS[t] if t >= 0 else "O"
        if tag == "O":
            if cur: spans.append(cur); cur = None
            continue
        p, lab = tag.split("-", 1)
        if p == "I" and cur and cur[2] == lab:
            cur = (cur[0], e, lab)
        else:
            if cur: spans.append(cur)
            cur = (s, e, lab)
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while e > s and (text[e - 1].isspace() or text[e - 1] in STRIP_TRAIL):
            e -= 1
        if e > s:
            out.append((s, e, lab))
    return out

# %% 5. STRICT MACRO-F1 (same idea as the official metric) + ORACLE ALIGNMENT CHECK
def strict_scores(pred, ref):
    """pred/ref: {doc: set((s, e, label))} -> macro P/R/F1 + per-label F1"""
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for d in ref:
        P, R = pred.get(d, set()), ref[d]
        for x in P & R: tp[x[2]] += 1
        for x in P - R: fp[x[2]] += 1
        for x in R - P: fn[x[2]] += 1
    per = {}
    for l in LABELS:
        p = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        r = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = (p, r, 2 * p * r / (p + r) if p + r else 0.0)
    mean = lambda i: float(np.mean([v[i] for v in per.values()]))
    return {"precision": mean(0), "recall": mean(1), "f1": mean(2), "per_label_f1": {l: round(v[2], 4) for l, v in per.items()}}

gold_excl = {d: [(*to_excl(s, e), lab) for s, e, lab, _ in ents] for d, ents in gold.items() if d in docs}

# Encode gold -> BIO -> decode back. Score = best strict F1 ANY model can reach with this tokenizer.
cache = {d: tokenize_with_offsets(docs[d]) for d in gold_excl}
ref_sets = {d: set(v) for d, v in gold_excl.items()}
best = None
for cand in ["", ".,;:"]:
    STRIP_TRAIL = cand
    orc = {d: set(decode_spans(docs[d], cache[d][1], bio_tags(docs[d], cache[d][1], gold_excl[d]))) for d in gold_excl}
    f1 = strict_scores(orc, ref_sets)["f1"]
    print(f"Oracle strict macro-F1 (strip trailing {cand!r}): {f1:.4f}")
    if best is None or f1 > best[0]:
        best = (f1, cand)
STRIP_TRAIL = best[1]
print(f"-> using STRIP_TRAIL={STRIP_TRAIL!r}. Oracle < 1.0 = entity boundaries inside a token "
      "(a tokenizer limit; note it in the report).")

# %% 6. WINDOWS (long documents -> overlapping windows; each token is trained/predicted ONCE,
#       in the window where it has the most left context - important for a causal model)
STEP = MAX_LEN - OVERLAP

def make_windows(ids, tags=None):
    out, start = [], 0
    if not ids:
        return out
    while True:
        end = min(start + MAX_LEN, len(ids))
        own_from = 0 if start == 0 else OVERLAP          # first OVERLAP tokens were owned by the previous window
        lab = [-100] * (end - start)
        if tags is not None:
            lab[own_from:] = tags[start + own_from:end]
        out.append({"ids": ids[start:end], "labels": lab, "start": start, "own_from": own_from})
        if end == len(ids):
            return out
        start += STEP

doc_ids = sorted(gold_excl)
random.Random(SEED).shuffle(doc_ids)
n_val = max(1, int(len(doc_ids) * VAL_FRAC))
val_ids, train_ids = doc_ids[:n_val], doc_ids[n_val:]
if QUICK_TEST:
    train_ids, val_ids = train_ids[:40], val_ids[:10]
    EPOCHS = 1

train_windows = []
for d in train_ids:
    ids, offs = cache[d]
    train_windows += make_windows(ids, bio_tags(docs[d], offs, gold_excl[d]))
lens = [len(cache[d][0]) for d in doc_ids]
print(f"train docs {len(train_ids)} | val docs {len(val_ids)} | train windows {len(train_windows)} | "
      f"doc length in tokens: median {int(np.median(lens))}, max {max(lens)}")

# %% 7. PHASE-1 MODEL DEFINITIONS (copied unchanged from the Phase-1 notebooks, one function per model)
def build_model1(cfg):
    # ---------- Model 1: GPT-style ----------
    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            assert d_model % n_head == 0
            self.d_model = d_model
            self.n_head = n_head
            self.head_dim = d_model // n_head
            self.c_attn = nn.Linear(d_model, 3 * d_model)
            self.c_proj = nn.Linear(d_model, d_model)
            self.attn_dropout = dropout
            self.resid_dropout = nn.Dropout(dropout)

        def forward(self, x):
            B, T, C = x.shape
            qkv = self.c_attn(x)
            q, k, v = qkv.split(self.d_model, dim=2)
            q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=(self.attn_dropout if self.training else 0.0),
                is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            y = self.c_proj(y)
            y = self.resid_dropout(y)
            return y


    class MLP(nn.Module):
        def __init__(self, d_model, dropout):
            super().__init__()
            self.fc = nn.Linear(d_model, 4 * d_model)
            self.proj = nn.Linear(4 * d_model, d_model)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x):
            x = self.fc(x)
            x = F.gelu(x)
            x = self.proj(x)
            x = self.dropout(x)
            return x


    class TransformerBlock(nn.Module):
        def __init__(self, d_model, n_head, dropout):
            super().__init__()
            self.ln_1 = nn.LayerNorm(d_model)
            self.attn = CausalSelfAttention(d_model, n_head, dropout)
            self.ln_2 = nn.LayerNorm(d_model)
            self.mlp = MLP(d_model, dropout)

        def forward(self, x):
            x = x + self.attn(self.ln_1(x))
            x = x + self.mlp(self.ln_2(x))
            return x


    class LegalGPT(nn.Module):
        def __init__(self, vocab_size, block_size, n_layer, n_head, d_model, dropout):
            super().__init__()
            self.block_size = block_size
            self.wte = nn.Embedding(vocab_size, d_model)
            self.wpe = nn.Embedding(block_size, d_model)
            self.blocks = nn.ModuleList([TransformerBlock(d_model, n_head, dropout) for _ in range(n_layer)])
            self.ln_f = nn.LayerNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.wte.weight
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            B, T = idx.shape
            assert T <= self.block_size
            positions = torch.arange(0, T, device=idx.device)
            x = self.wte(idx) + self.wpe(positions)
            for block in self.blocks:
                x = block(x)
            x = self.ln_f(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return LegalGPT(**cfg)


def build_model2(cfg):
    # ---------- Model 2: LLaMA-style ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class CausalSelfAttention(nn.Module):
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = CausalSelfAttention(d_model, n_heads, max_seq_len)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([LLaMABlock(cfg.d_model, cfg.n_heads, cfg.ffn_dim, cfg.max_seq_len)
                                         for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
        # (your generate() method is not needed: the pipeline's batched generate_batch() is used)
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model3(cfg):
    # ---------- Model 3: LLaMA + GQA + sliding window ----------
    class RMSNorm(nn.Module):
        """x / sqrt(mean(x^2) + eps) * weight  (no mean subtraction, unlike LayerNorm)"""
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return (x * rms) * self.weight


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles (relative position info)."""
        def __init__(self, dim, max_seq_len, theta=10000.0):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.size(-2)
            if seq_len > self.cos_cached.size(1):
                self._build_cache(seq_len)
            cos = self.cos_cached[:, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :seq_len, :].to(q.dtype)
            return apply_rotary(q, cos, sin), apply_rotary(k, cos, sin)


    def rotate_half(x):
        half = x.shape[-1] // 2
        x1, x2 = x[..., :half], x[..., half:]
        return torch.cat([-x2, x1], dim=-1)


    def apply_rotary(x, cos, sin):
        return x * cos + rotate_half(x) * sin


    class SwiGLU(nn.Module):
        """down( SiLU(gate(x)) * up(x) ), no bias"""
        def __init__(self, d_model, ffn_dim):
            super().__init__()
            self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
            self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)

        def forward(self, x):
            return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


    class GQASlidingWindowAttention(nn.Module):
        """
        GQA: n_heads query heads share n_kv_heads key/value heads (8 Q -> 2 KV: each KV head serves 4 Q heads).
        Sliding window: token i attends only to tokens j with 0 <= i - j < window_size (causal + local).
        """
        def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, window_size):
            super().__init__()
            assert d_model % n_heads == 0
            assert n_heads % n_kv_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.n_kv_heads = n_kv_heads
            self.head_dim = d_model // n_heads
            self.window_size = window_size
            self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
            self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
            # mask[q, k] = True -> q may attend to k   (k <= q  AND  q - k < window)
            positions = torch.arange(max_seq_len)
            distance = positions[:, None] - positions[None, :]
            self.register_buffer("window_mask", (distance >= 0) & (distance < window_size), persistent=False)

        def forward(self, x):
            B, T, C = x.shape
            q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)     # [B, 8, T, D]
            k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)  # [B, 2, T, D]
            q, k = self.rope(q, k)
            repeat_factor = self.n_heads // self.n_kv_heads
            k = k.repeat_interleave(repeat_factor, dim=1)                                  # [B, 8, T, D]
            v = v.repeat_interleave(repeat_factor, dim=1)
            mask = self.window_mask[:T, :T]
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class LLaMABlock(nn.Module):
        def __init__(self, d_model, n_heads, n_kv_heads, ffn_dim, max_seq_len, window_size):
            super().__init__()
            self.attn_norm = RMSNorm(d_model)
            self.attention = GQASlidingWindowAttention(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads,
                                                       max_seq_len=max_seq_len, window_size=window_size)
            self.ffn_norm = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model, ffn_dim)

        def forward(self, x):
            x = x + self.attention(self.attn_norm(x))   # pre-norm
            x = x + self.ffn(self.ffn_norm(x))
            return x


    class LLaMALanguageModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.cfg = cfg
            self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.blocks = nn.ModuleList([
                LLaMABlock(d_model=cfg.d_model, n_heads=cfg.n_heads, n_kv_heads=cfg.n_kv_heads, ffn_dim=cfg.ffn_dim,
                           max_seq_len=cfg.max_seq_len, window_size=cfg.window_size)
                for _ in range(cfg.n_layers)])
            self.final_norm = RMSNorm(cfg.d_model)
            self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, idx, targets=None):
            x = self.token_embedding(idx)          # no position embedding: RoPE handles position
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss
    return LLaMALanguageModel(SimpleNamespace(**cfg))


def build_model4(cfg):
    # ---------- Model 4: hybrid attention + gated conv ----------
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)


    class RotaryEmbedding(nn.Module):
        """RoPE: rotate q and k by position-dependent angles."""
        def __init__(self, dim, max_seq_len=2048, base=10000):
            super().__init__()
            self.dim = dim
            self.max_seq_len = max_seq_len
            self.base = base
            inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
            self.register_buffer("inv_freq", inv_freq, persistent=False)
            self._build_cache(max_seq_len)

        def _build_cache(self, seq_len):
            positions = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
            freqs = torch.outer(positions, self.inv_freq)
            emb = torch.cat([freqs, freqs], dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, None, :, :], persistent=False)
            self.register_buffer("sin_cached", emb.sin()[None, None, :, :], persistent=False)

        def forward(self, q, k):
            seq_len = q.shape[-2]
            if seq_len > self.cos_cached.shape[-2]:
                self._build_cache(seq_len)
            # FIX: cast to q's dtype. Otherwise fp16 q * fp32 cos -> fp32 q/k while v stays fp16,
            # and scaled_dot_product_attention can fail with a dtype mismatch.
            cos = self.cos_cached[:, :, :seq_len, :].to(q.dtype)
            sin = self.sin_cached[:, :, :seq_len, :].to(q.dtype)
            q = q * cos + rotate_half(q) * sin
            k = k * cos + rotate_half(k) * sin
            return q, k


    class RMSNorm(nn.Module):
        def __init__(self, dim, eps=1e-6):
            super().__init__()
            self.weight = nn.Parameter(torch.ones(dim))
            self.eps = eps

        def forward(self, x):
            rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
            return x * rms * self.weight


    class SwiGLU(nn.Module):
        def __init__(self, d_model, hidden_dim):
            super().__init__()
            self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
            self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
            self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

        def forward(self, x):
            return self.w2(F.silu(self.w1(x)) * self.w3(x))


    class CausalSelfAttention(nn.Module):
        """GLOBAL mixer: token i attends to tokens 0..i."""
        def __init__(self, d_model, n_heads, max_seq_len):
            super().__init__()
            assert d_model % n_heads == 0
            self.d_model = d_model
            self.n_heads = n_heads
            self.head_dim = d_model // n_heads
            self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)
            self.rope = RotaryEmbedding(self.head_dim, max_seq_len=max_seq_len)

        def forward(self, x):
            B, T, C = x.shape
            q, k, v = self.qkv(x).chunk(3, dim=-1)
            q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
            q, k = self.rope(q, k)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=True)
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            return self.out_proj(y)


    class GatedCausalConv(nn.Module):
        """
        LOCAL mixer: Linear(D->2D) -> (value, gate);
        value -> causal depthwise Conv1d (left padding only: position t sees t-k+1 .. t, never t+1);
        out = Linear( conv(value) * sigmoid(gate) ).
        """
        def __init__(self, d_model, kernel_size=5):
            super().__init__()
            self.d_model = d_model
            self.kernel_size = kernel_size
            self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
            self.conv = nn.Conv1d(d_model, d_model, kernel_size=kernel_size, groups=d_model, bias=False)
            self.out_proj = nn.Linear(d_model, d_model, bias=False)

        def forward(self, x):
            value, gate = self.in_proj(x).chunk(2, dim=-1)
            value = value.transpose(1, 2)                          # [B, D, T] for Conv1d
            value = F.pad(value, (self.kernel_size - 1, 0))        # LEFT padding only -> causal
            value = self.conv(value).transpose(1, 2)               # back to [B, T, D]
            return self.out_proj(value * torch.sigmoid(gate))


    class HybridBlock(nn.Module):
        def __init__(self, d_model, n_heads, ffn_dim, max_seq_len, conv_kernel, use_attention):
            super().__init__()
            self.use_attention = use_attention
            self.norm1 = RMSNorm(d_model)
            self.mixer = (CausalSelfAttention(d_model=d_model, n_heads=n_heads, max_seq_len=max_seq_len)
                          if use_attention else GatedCausalConv(d_model=d_model, kernel_size=conv_kernel))
            self.norm2 = RMSNorm(d_model)
            self.ffn = SwiGLU(d_model=d_model, hidden_dim=ffn_dim)

        def forward(self, x):
            x = x + self.mixer(self.norm1(x))
            x = x + self.ffn(self.norm2(x))
            return x


    class HybridLM(nn.Module):
        def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, max_seq_len, conv_kernel):
            super().__init__()
            self.vocab_size = vocab_size
            self.d_model = d_model
            self.token_embedding = nn.Embedding(vocab_size, d_model)
            # even layers -> attention, odd layers -> gated conv
            self.blocks = nn.ModuleList([
                HybridBlock(d_model=d_model, n_heads=n_heads, ffn_dim=ffn_dim, max_seq_len=max_seq_len,
                            conv_kernel=conv_kernel, use_attention=(i % 2 == 0))
                for i in range(n_layers)])
            self.final_norm = RMSNorm(d_model)
            self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
            self.lm_head.weight = self.token_embedding.weight      # weight tying
            self.apply(self._init_weights)

        def _init_weights(self, module):
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

        def forward(self, input_ids, targets=None):
            x = self.token_embedding(input_ids)
            for block in self.blocks:
                x = block(x)
            x = self.final_norm(x)
            logits = self.lm_head(x)
            loss = None
            if targets is not None:
                # FIX: the pipeline's targets are ALREADY shifted by one (y = x[1:]).
                # Your version shifted again (logits[:, :-1] vs targets[:, 1:]) -> the model
                # would learn to predict 2 tokens ahead. Same loss as Models 1-3 now.
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
    return HybridLM(**cfg)


BUILDERS = {"gpt": build_model1, "llama": build_model2, "gqa": build_model3, "hybrid": build_model4}

# %% 8. LOAD PHASE-1 WEIGHTS + NER HEAD
if PHASE1_CKPT is None:
    print(".pt files under /kaggle/input:", glob.glob("/kaggle/input/**/*.pt", recursive=True))
    raise SystemExit("Set PHASE1_CKPT in Cell 1 to one of the paths above")
def as_ckpt_file(path):
    """Kaggle sometimes auto-extracts an uploaded .pt (it is a zip inside) into a FOLDER.
    If `path` is such a folder, re-pack it into a normal .pt file that torch.load can read."""
    if not os.path.isdir(path):
        return path
    root = next((r for r, _, f in os.walk(path) if "data.pkl" in f), None)
    assert root, f"{path} is a folder without data.pkl - not an extracted checkpoint"
    out = os.path.join("/kaggle/working", os.path.basename(path.rstrip("/")) + "_repacked.pt")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for r, _, files in os.walk(root):
            for f in files:
                full = os.path.join(r, f)
                z.write(full, "archive/" + os.path.relpath(full, root))
    print(f"Re-packed extracted checkpoint folder {path} -> {out}")
    return out

PHASE1_CKPT = as_ckpt_file(PHASE1_CKPT)
ck = torch.load(PHASE1_CKPT, map_location="cpu")
cfg = ck["config"]
print("Phase-1 config:", cfg)
assert cfg["vocab_size"] == len(tokenizer), "Tokenizer doesn't match this Phase-1 model (vocab size differs)"
backbone = BUILDERS[MODEL_TYPE](cfg)
backbone.load_state_dict(ck["model"])
ctx = cfg.get("block_size", cfg.get("max_seq_len"))
assert MAX_LEN <= ctx, f"MAX_LEN {MAX_LEN} > Phase-1 context {ctx}"
D_MODEL = next(m for m in backbone.modules() if isinstance(m, nn.Embedding)).weight.shape[1]
backbone.lm_head = nn.Identity()   # forward() now returns hidden states instead of vocab logits


class NERModel(nn.Module):
    def __init__(self, backbone, d_model, n_tags):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(d_model, n_tags)

    def forward(self, ids):
        h, _ = self.backbone(ids)            # (B, T, d_model)
        return self.head(self.dropout(h))   # (B, T, n_tags)

model = NERModel(backbone, D_MODEL, len(TAGS))

teacher = None
if KD_TEACHER:
    # teacher = a fine-tuned Phase-2 NER model (frozen); student = smaller copy of the Phase-1 model
    t_type = KD_TEACHER_TYPE or MODEL_TYPE
    t_cfg = torch.load(as_ckpt_file(KD_TEACHER_CKPT) if KD_TEACHER_CKPT else PHASE1_CKPT, map_location="cpu")["config"]
    t_bb = BUILDERS[t_type](t_cfg); t_bb.lm_head = nn.Identity()
    t_d = next(m for m in t_bb.modules() if isinstance(m, nn.Embedding)).weight.shape[1]
    teacher = NERModel(t_bb, t_d, len(TAGS))
    teacher.load_state_dict(torch.load(as_ckpt_file(KD_TEACHER), map_location="cpu"))
    teacher = teacher.to(DEVICE).eval()
    for p_ in teacher.parameters(): p_.requires_grad = False
    n_teacher = sum(p_.numel() for p_ in teacher.parameters())
    # student keeps every KD_KEEP_EVERY-th block, initialised from the Phase-1 weights (DistilBERT-style)
    model.backbone.blocks = nn.ModuleList(list(model.backbone.blocks)[::KD_KEEP_EVERY])
    n_student = sum(p_.numel() for p_ in model.parameters())
    print(f"KD: teacher {n_teacher:,} params -> student {n_student:,} params "
          f"({len(model.backbone.blocks)} blocks, {100 * n_student / n_teacher:.0f}% of teacher)")

# %% 9. FINE-TUNING METHODS
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A x)          - additive low-rank update (LoRA)"""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))   # zero -> starts as the original layer

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)


class LoRMALinear(nn.Module):
    """y = (I + (alpha/r) B A) W x           - MULTIPLICATIVE low-rank update of W (LoRMA, identity variant).
       Check the exact formulation (and the rank-inflation variants) against the LoRMA paper before the report."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))   # zero -> (I + 0) W = W at start

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)


class AdapterBlock(nn.Module):
    """Houlsby-style bottleneck adapter after a frozen Phase-1 block: x + up(GELU(down(x)))"""
    def __init__(self, block, d_model, dim):
        super().__init__()
        self.block = block
        self.down, self.up = nn.Linear(d_model, dim), nn.Linear(dim, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)   # starts as identity

    def forward(self, x):
        x = self.block(x)
        return x + self.up(F.gelu(self.down(x)))


def wrap_linears(module, wrapper):
    for name, child in module.named_children():
        if isinstance(child, nn.Linear):
            setattr(module, name, wrapper(child))
        else:
            wrap_linears(child, wrapper)

bb = model.backbone
if FT_METHOD == "full":
    pass
elif FT_METHOD == "partial":
    for p in bb.parameters(): p.requires_grad = False
    for blk in bb.blocks[FREEZE_BOTTOM:]:
        for p in blk.parameters(): p.requires_grad = True
    for name, p in bb.named_parameters():            # final norm stays trainable
        if not name.startswith("blocks.") and p.dim() == 1: p.requires_grad = True
elif FT_METHOD == "bitfit":
    # biases + 1-D norm gains only. NOTE: Models 2-4 have no biases, so for them this trains only norm weights.
    for name, p in bb.named_parameters():
        p.requires_grad = p.dim() == 1
elif FT_METHOD in ("lora", "lorma"):
    for p in bb.parameters(): p.requires_grad = False
    W = LoRALinear if FT_METHOD == "lora" else LoRMALinear
    for blk in bb.blocks:
        wrap_linears(blk, lambda lin: W(lin, LORA_R, LORA_ALPHA))
elif FT_METHOD == "adapter":
    for p in bb.parameters(): p.requires_grad = False
    bb.blocks = nn.ModuleList([AdapterBlock(b, D_MODEL, ADAPTER_DIM) for b in bb.blocks])
else:
    raise ValueError(FT_METHOD)

model = model.to(DEVICE)
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{FT_METHOD}: trainable {n_train:,} / {n_total:,} params ({100 * n_train / n_total:.2f}%)")

# %% 10. TRAINING
new_param_ids = {id(p) for n, p in model.named_parameters()
                 if n.startswith("head.") or any(k in n for k in (".A", ".B", "down.", "up."))
                 or FT_METHOD == "bitfit"}
groups = [
    {"params": [p for p in model.parameters() if p.requires_grad and id(p) in new_param_ids], "lr": LR_NEW},
    {"params": [p for p in model.parameters() if p.requires_grad and id(p) not in new_param_ids], "lr": LR_BODY},
]
optimizer = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * math.ceil(len(train_windows) / BATCH_SIZE)
warmup = max(1, int(0.06 * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min(1.0, (s + 1) / warmup) * max(0.0, (total_steps - s) / max(1, total_steps - warmup)))

USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
autocast = lambda: torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)
class_w = torch.ones(len(TAGS), device=DEVICE); class_w[TAG2ID["O"]] = O_WEIGHT

def collate(batch):
    T = max(len(w["ids"]) for w in batch)
    ids = torch.zeros(len(batch), T, dtype=torch.long)
    lab = torch.full((len(batch), T), -100, dtype=torch.long)
    for i, w in enumerate(batch):
        # right padding is safe for a causal model: real tokens never attend to the padding after them
        ids[i, :len(w["ids"])] = torch.tensor(w["ids"])
        lab[i, :len(w["labels"])] = torch.tensor(w["labels"])
    return ids.to(DEVICE), lab.to(DEVICE)

@torch.no_grad()
def predict_doc(text):
    ids, offs = tokenize_with_offsets(text)
    tags = [0] * len(ids)
    model.eval()
    wins = make_windows(ids)
    for s in range(0, len(wins), BATCH_SIZE):
        chunk = wins[s:s + BATCH_SIZE]
        x, _ = collate(chunk)
        with autocast():
            pred = model(x).argmax(-1).tolist()
        for w, p in zip(chunk, pred):
            for k in range(w["own_from"], len(w["ids"])):
                tags[w["start"] + k] = p[k]
    return decode_spans(text, offs, tags)

def evaluate(doc_list):
    pred = {d: set(predict_doc(docs[d])) for d in doc_list}
    return strict_scores(pred, {d: set(gold_excl[d]) for d in doc_list})

BEST_PATH = os.path.join(OUTPUT_DIR, "best_ner.pt")
best_f1, history, t0 = -1.0, [], time.time()
if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_windows)
    tot, n = 0.0, 0
    for s in range(0, len(train_windows), BATCH_SIZE):
        x, y = collate(train_windows[s:s + BATCH_SIZE])
        with autocast():
            logits = model(x)
        loss = F.cross_entropy(logits.float().reshape(-1, len(TAGS)), y.reshape(-1), weight=class_w, ignore_index=-100)
        if teacher is not None:
            with torch.no_grad(), autocast():
                t_logits = teacher(x)
            mask = y != -100
            kl = F.kl_div(F.log_softmax(logits.float() / KD_T, -1), F.softmax(t_logits.float() / KD_T, -1),
                          reduction="none").sum(-1)
            loss = KD_ALPHA * loss + (1 - KD_ALPHA) * (KD_T ** 2) * kl[mask].mean()
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer); scaler.update(); sched.step()
        tot += loss.item(); n += 1
    res = evaluate(val_ids)
    history.append({"epoch": epoch, "train_loss": tot / n, **{k: res[k] for k in ("precision", "recall", "f1")}})
    print(f"epoch {epoch} | train loss {tot / n:.4f} | val strict macro P {res['precision']:.4f} "
          f"R {res['recall']:.4f} F1 {res['f1']:.4f} | {time.time() - t0:.0f}s")
    if res["f1"] > best_f1:
        best_f1, best_res = res["f1"], res
        torch.save({k: v for k, v in model.state_dict().items()}, BEST_PATH)
        print("  >> best model saved")

TRAIN_MIN = (time.time() - t0) / 60
PEAK_GB = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else None
print(f"\nBest val strict macro-F1 {best_f1:.4f} | train time {TRAIN_MIN:.1f} min | peak GPU mem {PEAK_GB} GB")
print("Per-label F1 (best epoch):", best_res["per_label_f1"])

# %% 11. TEST PREDICTIONS + prediction.json + ZIP
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
predictions = {}
for i, (d, text) in enumerate(test_docs.items()):
    predictions[d] = [[*from_excl(s, e), lab] for s, e, lab in predict_doc(text)]
    if (i + 1) % 50 == 0: print(f"{i + 1}/{len(test_docs)}")

assert set(predictions) == set(test_docs), "doc IDs don't match the test file"
assert all(lab in LABELS for v in predictions.values() for _, _, lab in v)
print(f"{len(predictions)} docs | {sum(len(v) for v in predictions.values())} entities predicted")
print("Example:", next(iter(predictions.items()))[1][:5])

pred_path = os.path.join(OUTPUT_DIR, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False)
zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")
print("Submission ready:", zip_path)

# %% 12. SAVE RUN SUMMARY (experiment log / report)
with open(os.path.join(OUTPUT_DIR, "run_summary.json"), "w") as f:
    json.dump({"model_type": MODEL_TYPE, "phase1_ckpt": PHASE1_CKPT, "tokenizer": TOKENIZER,
               "ft_method": FT_METHOD, "kd_teacher": KD_TEACHER,
               "kd_teacher_params": n_teacher if KD_TEACHER else None, "kd_keep_every": KD_KEEP_EVERY,
               "kd_T": KD_T, "kd_alpha": KD_ALPHA, "trainable_params": n_train, "total_params": n_total,
               "max_len": MAX_LEN, "overlap": OVERLAP, "epochs": EPOCHS, "lr_body": LR_BODY, "lr_new": LR_NEW,
               "o_weight": O_WEIGHT, "lora_r": LORA_R, "adapter_dim": ADAPTER_DIM, "freeze_bottom": FREEZE_BOTTOM,
               "end_inclusive": END_INCLUSIVE, "strip_trail": STRIP_TRAIL, "oracle_f1": best[0],
               "val_best": best_res, "history": history,
               "train_minutes": round(TRAIN_MIN, 1), "peak_gpu_gb": PEAK_GB}, f, indent=2)
print("Done.")

Device: cuda Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/tmp/cs781_data/Phase_2_LNER/Train/labels.txt 0.0 MB
/tmp/cs781_data/Phase_2_LNER/Train/fold2-text.json 1.1 MB
/tmp/cs781_data/Phase_2_LNER/Train/fold1-gold.json 0.5 MB
/tmp/cs781_data/Phase_2_LNER/Train/fold2-gold.json 0.5 MB
/tmp/cs781_data/Phase_2_LNER/Train/fold1-text.json 1.1 MB
/tmp/cs781_data/Phase_2_LNER/Test/fold3-text.json 1.2 MB

Train/fold1-text.json: dict with 35 entries | first: ["115651329", "REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Cour...

Train/fold1-gold.json: dict with 35 entries | first: ["115651329", [[137, 153, "RESP", "STATE OF HARYANA"], [252, 261, "DATE", "19.8.2011"], [276, 308, "COURT", "High Court of Punjab and Haryana"], [359, 369, "COURT", "High Court"], [575, 585, "DATE", "18.11.1994"], [637, 644, "WIT", "Jagdish"], [6